# Summer Analytics 2026 - E-Commerce Conversion Prediction
## Final Submission Notebook

This notebook contains the complete, reproducible pipeline used to generate the final `submission.csv` for the private test dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Load the Datasets
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('private_test.csv')

# 2. Separate Features and Target
X = train_df.drop(['User_ID', 'Converted'], axis=1)
y = train_df['Converted']
X_test = test_df.drop(['User_ID'], axis=1)
test_ids = test_df['User_ID']

In [ ]:
# 3. Configure Preprocessing Pipeline
numerical_cols = ['Age', 'Income', 'Pages_Viewed', 'Products_Viewed', 'Time_On_Site', 'Previous_Purchases', 'Browser_Version', 'Campaign_Code']
categorical_cols = ['City_Tier', 'Device_Type', 'Traffic_Source', 'Discount_Seen']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [ ]:
# 4. Define the Model Pipeline
# Using Random Forest with balanced class weights to maximize the F1-Score
model = RandomForestClassifier(n_estimators=200, 
                               max_depth=10, 
                               random_state=42, 
                               class_weight='balanced')

clf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('model', model)])

In [ ]:
# 5. Train Model on Full Training Data
clf.fit(X, y)

# 6. Generate Predictions for private_test.csv
final_predictions = clf.predict(X_test)

# 7. Export to submission.csv
submission_df = pd.DataFrame({
    'User_ID': test_ids,
    'Converted': final_predictions
})

submission_df.to_csv('submission.csv', index=False)
print("Training complete. 'submission.csv' has been successfully generated in the root directory!")